---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">Sentimento: e se usássemos um LLM?</h1>

<h3 align="center">Ângelo Rosa</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
</div>

---

## 1. A pergunta

O classificador do notebook anterior chegou a 0,663 de F1-macro. Não é um resultado brilhante, e a
pergunta aparece sozinha: **por que não pedir a um LLM que leia cada review e diga o sentimento?**
Ele entende linguagem muito melhor que um XGBoost sobre embeddings.

A resposta curta é: porque custa caro e demora. Este notebook mede quanto, com dois exemplos e uma
conta de padaria honesta.

Uma ressalva antes de começar: dois exemplos **não** medem a qualidade do LLM. Para isso seria
preciso rodar o conjunto de teste inteiro, que é exatamente o que estamos evitando. O que dois
exemplos medem bem é o **custo unitário**, e é com ele que a conta é feita.

In [1]:
import time

import joblib
import numpy as np
import pandas as pd
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from sklearn.model_selection import train_test_split

from config import EMBEDDING_DIR, GROQ_MODEL, MODELS_DIR, SEED, SENTIMENT_LABELS, TEST_SIZE

emb = np.load(f"{EMBEDDING_DIR}/emb.npy")
meta = pd.read_parquet(f"{EMBEDDING_DIR}/meta.parquet")
y = meta["sentiment_label"].to_numpy()
textos = meta["text_prep"].fillna("").to_numpy()

_, X_teste, _, y_teste, _, txt_teste = train_test_split(
    emb, y, textos, test_size=TEST_SIZE, stratify=y, random_state=SEED)

modelo = joblib.load(f"{MODELS_DIR}/sentiment_model.joblib")
pred = modelo.predict(X_teste)

# um neutro que o XGBoost errou e um negativo que ele acertou
exemplos = [np.where((y_teste == 1) & (pred != 1))[0][0],
            np.where((y_teste == 0) & (pred == 0))[0][0]]

llm = ChatGroq(model=GROQ_MODEL, temperature=0, max_tokens=2048)
PROMPT = ("Classifique o sentimento deste review em uma palavra: negativo, neutro ou positivo.\n"
          "Review: {texto}\nResposta:")

linhas = []
for i in exemplos:
    inicio = time.time()
    resposta = llm.invoke([HumanMessage(PROMPT.format(texto=txt_teste[i]))])
    uso = resposta.response_metadata["token_usage"]
    linhas.append({
        "review": txt_teste[i][:55] + "...",
        "real": SENTIMENT_LABELS[y_teste[i]],
        "XGBoost": SENTIMENT_LABELS[pred[i]],
        "LLM": resposta.content.strip().lower(),
        "tokens entrada": uso["prompt_tokens"],
        "tokens saída": uso["completion_tokens"],
        "segundos": round(time.time() - inicio, 2),
    })

resultados = pd.DataFrame(linhas)
resultados

,review,real,XGBoost,LLM,tokens entrada,tokens saída,segundos
0,Cleans well. I have a hard time using the butt...,neutro,positivo,neutro,114,147,0.9
1,Not as described!!. I wouldn't have chosen to ...,negativo,negativo,negativo,164,77,0.5


Três coisas na tabela.

**O LLM acertou o caso difícil.** O primeiro review é um neutro que o XGBoost chamou de positivo:
*"Cleans well. I have a hard time using the buttons due to my age and arthritis"*. O elogio vem
primeiro, a ressalva depois, e o LLM leu a frase inteira. É um caso, não uma medição, mas aponta
para onde está a vantagem dele: texto ambivalente, justamente a classe neutra.

**Uma palavra de resposta custa mais de cem tokens.** A saída foi "neutro", mas o modelo gastou 147
tokens de saída para chegar lá, porque raciocina antes de responder, e você paga por esse raciocínio
como paga por qualquer token gerado.

**Meio segundo a um segundo por review.** Parece pouco, e é o que transforma a conta da próxima
seção.

## 2. Quanto custaria classificar tudo

A conta usa o consumo medido acima, multiplicado pelos reviews da base, com o preço em vigor do
modelo (consulte `console.groq.com/docs/models`, os preços mudam). Do outro lado, o XGBoost
classificando os mesmos reviews na sua máquina.

In [2]:
PRECO_ENTRADA, PRECO_SAIDA = 0.15, 0.60   # dólares por milhão de tokens
N = len(meta)

entrada, saida = resultados["tokens entrada"].mean(), resultados["tokens saída"].mean()
custo = N * (entrada * PRECO_ENTRADA + saida * PRECO_SAIDA) / 1e6
horas = N * resultados["segundos"].mean() / 3600

inicio = time.time()
modelo.predict(emb)
segundos_xgb = time.time() - inicio

pd.DataFrame({
    f"LLM ({GROQ_MODEL})": [f"US$ {custo:,.2f}", f"{horas:,.0f} horas"],
    "XGBoost (local)": ["US$ 0,00", f"{segundos_xgb:.1f} segundos"],
}, index=[f"classificar os {N:,} reviews", "tempo, uma chamada por vez"])

,LLM (openai/gpt-oss-120b),XGBoost (local)
"classificar os 111,851 reviews",US$ 9.85,"US$ 0,00"
"tempo, uma chamada por vez",22 horas,0.1 segundos


## 3. O que a conta ensina

**US$ 9,85 e 22 horas** contra **US$ 0,00 e 0,1 segundo**. Vale ler esses números com cuidado,
porque o argumento não é o que parece à primeira vista.

Dez dólares não é proibitivo. Se o trabalho fosse classificar esta base uma vez, o LLM seria uma
opção perfeitamente razoável. O problema é outro: **esse custo é recorrente e linear**. Cada review
novo custa de novo, para sempre, enquanto o XGBoost foi treinado uma vez e classifica os 111.851 em
um décimo de segundo, offline, de graça. A diferença de tempo é de quase seis ordens de grandeza, e
é ela que decide arquitetura: com o LLM, classificar sentimento vira uma chamada de rede, com fila,
limite de uso, indisponibilidade e preço que muda sem te avisar.

**Onde o LLM ganha, então?** Em volume pequeno, em problema sem dado rotulado, e num papel que este
notebook não explorou: **rotular**. Usar o LLM para etiquetar alguns milhares de exemplos e treinar
o modelo clássico em cima deles costuma ser o melhor dos dois mundos, porque paga o custo uma vez e
fica com um modelo local.

**E o modelo do notebook 04, dá para melhorar?** Dá, por dois caminhos:

- **Melhorar o que já existe**: o rótulo vem da nota, e a seção 5 do notebook 04 mostrou que ele é
  ambíguo justamente no neutro. Rótulo melhor vale mais que algoritmo melhor.
- **Fine-tuning de um SLM**: pegar um modelo de linguagem pequeno e treiná-lo nesta tarefa. Fica com
  boa parte da compreensão de texto do LLM e volta a rodar local, sem custo por chamada. É o
  meio-termo entre os dois extremos deste notebook.

A moral não é que LLM é caro nem que modelo clássico é melhor. É que **eles resolvem problemas
diferentes**, e a conta de padaria acima é o que separa os dois em qualquer projeto real.